# Imports

In [ ]:
from Fcts_FE import make_experiment, merge_feature_tables_from_zarr
from Fcts_Base import get_stainings, find_zarr_dirs

%load_ext autoreload
%autoreload 2

# User Input

In [ ]:
# Point to experiment folder. Has to include setup .xls file.
source = "YOUR PATH TO EXPERIMENT FOLDER"
analysis_dir = None
experiment_ID = "YOUR EXPERIMENT ID"

# Feature table(s) to load (str or list[str])
feature_table_names = "YOUR_FEATURE_TABLE_NAME"

# Multiplexing round(s) to load from <row>/<col>/<round>/...
# Either int (single round) or list[int] (multiple rounds).
multiplexing_rounds = 0

# ROI table + label for downstream image/mask lookup
roi_table_name = "nuclei_ROI_table"
label_name = "nuclei"

file_ending = ".zarr"
result_file_name = "1_FeatureLoading"

# Load files and display experimental setup

In [ ]:
stainings = get_stainings(source)
folder = find_zarr_dirs(source, file_ending=file_ending)
experiment_setup, barcodes = make_experiment(source)

if analysis_dir is None:
    analysis_dir = source

# Load + merge features (and save)

In [ ]:
# Normalize multiplexing_rounds input
if isinstance(multiplexing_rounds, int):
    multiplexing_rounds = [multiplexing_rounds]
if len(multiplexing_rounds) == 0:
    raise ValueError('multiplexing_rounds is empty. Provide at least one round id.')

# Load one AnnData per round and concatenate along vars (features).
# Requires identical objects/order across rounds (same segmentation).
ad_list = []
for r in multiplexing_rounds:
    ad_r = merge_feature_tables_from_zarr(
        source=source,
        folder=folder,
        experiment_setup=experiment_setup,
        stainings=stainings,
        experiment_ID=experiment_ID,
        result_file_name=result_file_name,
        analysis_dir=analysis_dir,
        feature_table_names=feature_table_names,
        roi_table_name=roi_table_name,
        label_name=label_name,
        multiplexing_round=r,
    )
    ad_list.append(ad_r)

# If multiple rounds: concatenate features horizontally into a single AnnData
if len(ad_list) == 1:
    ad_all = ad_list[0]
else:
    # Validate identical obs_names
    obs0 = ad_list[0].obs_names
    for j, ad_r in enumerate(ad_list[1:], start=1):
        if not obs0.equals(ad_r.obs_names):
            raise ValueError(f'obs_names mismatch between rounds {multiplexing_rounds[0]} and {multiplexing_rounds[j]}')
    ad_all = ad.concat(ad_list, axis=1, merge='same', join='outer')

print('Merged AnnData ready:')
print('  n_obs:', ad_all.n_obs)
print('  n_vars:', ad_all.n_vars)
print('  rounds:', multiplexing_rounds)